In [ ]:
import json
import pickle
import argparse


import numpy as np
import matplotlib.pyplot as plt

from utils import *

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import copy

from data_loading import *

import json

import pandas as pd
from pathlib import Path

import sys
import os

from models import *
from approaches.standardized_residuals import StandardizedResiduals
import time

# from covmetrics import ERT
# from tabicl import TabICLClassifier

In [ ]:
parameters = {
    "load_name": "scm1d",
    "hidden_dim": 128,
    "num_layers": 3,
    "num_flows":1,
    "num_layers_flows": 4,
    "batch_size": 256,
    "num_epochs": 300,
    "lr": 1e-3,
}

num_epochs = parameters["num_epochs"]
batch_size = parameters["batch_size"]
lr         = parameters["lr"]
hidden_dim = parameters["hidden_dim"]
num_layers = parameters["num_layers"]
num_flows  = parameters["num_flows"]
num_layers_flows = parameters["num_layers_flows"]

tau = 0.9
alpha = 1-tau

In [ ]:
load_path = "../../data/processed_data/" + parameters["load_name"] + ".npz"
X, Y = load_data(load_path)

# splits = [0.7, 0.1, 0.1, 0.1]
splits = [0.3, 0.1, 0.5, 0.1]

dtype = torch.float32

subsets = split_and_preprocess(X, Y, splits=splits)

x_train, y_train, x_calibration, y_calibration, x_test, y_test, x_stop, y_stop = subsets["X_train"], subsets["Y_train"], subsets["X_calibration"], subsets["Y_calibration"], subsets["X_test"], subsets["Y_test"], subsets["X_stop"], subsets["Y_stop"]

print("X_train shape:", x_train.shape, "Y_train shape:", y_train.shape)
print("X_cal shape:", x_calibration.shape, "Y_cal shape:", y_calibration.shape)
print("X_test shape:", x_test.shape, "Y_test shape:", y_test.shape)
print("X_stop shape:", x_stop.shape, "Y_stop shape:", y_stop.shape)

input_dim = x_train.shape[1]
output_dim = y_train.shape[1]

dtype = torch.float32

x_train_tensor = torch.tensor(x_train, dtype=dtype)
y_train_tensor = torch.tensor(y_train, dtype=dtype)
x_stop_tensor = torch.tensor(x_stop, dtype=dtype)
y_stop_tensor = torch.tensor(y_stop, dtype=dtype)
x_calibration_tensor = torch.tensor(x_calibration, dtype=dtype)
y_calibration_tensor = torch.tensor(y_calibration, dtype=dtype)
x_test_tensor = torch.tensor(x_test, dtype=dtype)
y_test_tensor = torch.tensor(y_test, dtype=dtype)

In [ ]:
standardized_residuals = StandardizedResiduals(input_dim, 
                                                output_dim,
                                                hidden_dim = hidden_dim,
                                                num_layers = num_layers
                                                )

standardized_residuals.fit(X_train=x_train_tensor, 
                    y_train=y_train_tensor,
                    X_val = x_stop_tensor,
                    y_val = y_stop_tensor,
                    num_epochs=1000,
                    batch_size=batch_size,
                    lr=lr,
                    verbose = 1
                    )

standardized_residuals.conformalize(x=x_calibration_tensor, y=y_calibration_tensor, alpha = alpha)
coverage = standardized_residuals.get_coverage(x_test_tensor, y_test_tensor)
cover_standardized = standardized_residuals.get_cover(x_test_tensor, y_test_tensor)
volumes  = standardized_residuals.get_average_volume(x_test_tensor, scaled=True)

print("Coverage:", coverage)
print("Average Volume:", volumes)

In [ ]:
from tabicl import TabICLRegressor

lr = 5e-4

num_epochs = 250
batch_size = 100

tau = 1-alpha

tau_param_init = TauParameterAnnealer(tau,                
                warm_start_step=float('inf')
                )

tau_param_fine_tune = TauParameterAnnealer(tau,                
                warm_start_step=1, 
                tau_low_target_step=100, 
                tau_low_steepness=1e-3,
                tau_high_target_step=100, 
                tau_high_steepness=1e-2,
                low_error_init=0.5,   
                low_error_max=0.03,    
                high_error_init=0.2,  
                high_error_max=0.03,  
                eps=1e-5              
                )

model = UnifiedConditionalEstimator(dim_X=input_dim, dim_y=output_dim, 
                                    cov_mode="low_rank", num_flow_layers=3, K=1,
                                    det_normalized=False
                                    )

model.fit(x_train_tensor, y_train_tensor, 
        X_val = x_stop_tensor,
        y_val = y_stop_tensor,
        tau=tau, 
        epochs=num_epochs, 
        lr=lr, 
        batch_size=batch_size, 
        return_best=True, 
        print_every=1,
        tau_parameterAnnealer=tau_param_init,
        loss_function="log_volume"
        )


model.conformalize(x_calibration_tensor, y_calibration_tensor, tau)
average_volume = model.compute_average_volume(x_test_tensor, scaled=True)
print("Averaged volume : ", average_volume)


model.fit(x_train_tensor, y_train_tensor, 
        X_val = x_stop_tensor,
        y_val = y_stop_tensor,
        tau=tau, 
        epochs=num_epochs, 
        lr=lr, 
        batch_size=batch_size, 
        return_best=True, 
        print_every=100,
        tau_parameterAnnealer=tau_param_fine_tune,
        loss_function="log_volume"
        )


model.conformalize(x_calibration_tensor, y_calibration_tensor, tau)
average_volume = model.compute_average_volume(x_test_tensor, scaled=True)
print("Averaged volume : ", average_volume)

model.fit_quantile(x_train_tensor, y_train_tensor, 
        tau=tau, 
        X_val = x_stop_tensor,
        y_val = y_stop_tensor,
        epochs=num_epochs, 
        lr=lr, 
        batch_size=batch_size, 
        return_best=True, 
        print_every=100,
        )

model.conformalize(x_calibration_tensor, y_calibration_tensor, tau)
average_volume = model.compute_average_volume(x_test_tensor, scaled=True)
print("Averaged volume : ", average_volume)

# model.fit_external_quantile(TabICLRegressor(), x_train_tensor, y_train_tensor)
model.fit_external_quantile(TabICLRegressor(), x_stop_tensor, y_stop_tensor)

model.conformalize(x_calibration_tensor, y_calibration_tensor, tau)
average_volume = model.compute_average_volume(x_test_tensor, scaled=True)
print("Averaged volume : ", average_volume)

In [ ]:
model.fit_external_quantile(TabICLRegressor(), x_stop_tensor, y_stop_tensor)

model.conformalize(x_calibration_tensor, y_calibration_tensor, tau)
average_volume = model.compute_average_volume(x_test_tensor, scaled=True)
print("Averaged volume : ", average_volume)

In [ ]:
from data_plot import *
plot_combined_conditional_contours_test_multiflow(x_test_tensor, y_test_tensor, standardized_residuals, model, n_plots=5)

In [ ]:
plot_combined_conditional_contours_test_multiflow(x_test_tensor, y_test_tensor, standardized_residuals, model, n_plots=5)

In [ ]:
from covmetrics import ERT
from tabicl import TabICLClassifier

alpha = 1-tau
# ert_standardized = ERT(TabICLClassifier).evaluate(x_test_tensor, cover_standardized, alpha)
ert_level_sets = ERT().evaluate(x_test_tensor, cover_level_sets, alpha)
print('ert_level_sets', ert_level_sets)

# print("Stand", ert_standardized, "Level sets, ", ert_level_sets)

In [ ]:
ert_level_sets = ERT(TabICLClassifier).evaluate(x_test_tensor, cover_level_sets, alpha)

print("Stand", ert_standardized, "Level sets, ", ert_level_sets)